In [1]:
# Cryptogram Project
# Author: Althea Doherty
# Description: creating a cryptogram Puzzle with topics about movies, books, and music
#This game allows users to see encrypted text quotations in various themes: My Favorite Movies, Music, and Books/Manga 
#Letters in the cipher text represent unknown letters in the plain text (mono-alphabetic substitution cipher). 
#Users have to study the cipher text and input characters in the entry boxes to reveal the original quotation before losing their health points.

In [27]:
import random
import tkinter as tk

# Variables
PUZLVL = {
    "0: Rules": [
        """How to Play Cryptogram
Objective
Decode the encrypted quote or message by uncovering the hidden letters.

The Rules
1. One-to-One Substitution
Each letter in the puzzle stands for a different letter in the alphabet (if A represents R, every A in the puzzle is R).

2. No Self-Substitution
A letter will never represent itself in the solution (A will never decrypt to A).

3. Color Feedback
Green: Indicates a correct letter choice.
Red: Indicates an incorrect letter choice.

4. Punctuation & Numbers
All spaces, punctuation marks (commas, periods, apostrophes), and numbers remain unchanged and are given to you as clues.

5. Case Insensitivity
Letter replacements apply equally regardless of capitalization."""
    ],
    "1: Movie Quotes": [
        (
            "OHANA MEANS FAMILY",
            "RKDQD PHDQV IDPLOB",
        ),  # Lilo & Stitch
        (
            "THE BROKEN ARE MORE EVOLVED",
            "WKH EURNHQ DUH PRUH HYROYHG",
        ),  # Split
        (
            "THERE IS NO SPOON",
            "WKHUH LV QR VSRRQ",
        ),  # The Matrix
    ],
    "2: Music Lyrics": [
        (
            "IM AN EDUCATED FOOL WITH MONEY ON MY MIND",
            "JP ER IHXDEWIX IRRO ZPWP PRRPC RP PR PJRI",
        ),  # Coolio - Gangsta's Paradise
        (
            "ITS THE COLORS YOU HAVE NO NEED TO BE SAD",
            "JWV WKH FRORUV BRX KDYH QR QHHG WR EH VDG",
        ),  # Group Love - Colours
        (
            "WITHIN HIS DREAMS HE SEES THE LIFE HE MADE",
            "ZLWKLQ HLV GUHDPV KH VHHV WKH OLIH KH PDGH",
        ),  # Kid Cudi - Day'N'Nite
    ],
    "3: Book & Manga Quotes": [
        (
            "THIS IS MY HOLE IT WAS MADE FOR ME",
            "WKLV LV PB KROH LW ZDV PDGH IRU PH",
        ),  # The Enigma of Amigara Fault by Junji Ito
        (
            "IT WAS A PLEASURE TO BURN",
            "LW ZDV D SOHDVXUH WR EXUQ",
        ),  # Fahrenheit 451 by Ray Bradbury
    ],
}

# Main Window
window = tk.Tk()
window.geometry("850x550")
window.title("Cryptogram Puzzle")

health = tk.IntVar(value=100)
lvlselect = tk.StringVar(value="0: Rules")  # Default difficulty

# Global game state variables
correct = ""
encrypted = ""
entries = []        # Stores tuple: (ogIndx, eWidget)
corAns = []         # Stores list of uppercase letters in correct quote

# Window Set Up Functions & Validations
def validChar(newVal):
    if newVal == "":
        return True
    return len(newVal) == 1 and newVal.isalpha()

vcmd = (window.register(validChar), "%P")

def hLvl(amount):
    hUpdate = max(0, health.get() - amount)
    health.set(hUpdate)
    hLabel.config(text=f"Health: {hUpdate}")

def loadLevel(key):
    global correct, encrypted, corAns, entries

    # Reset health
    health.set(100)
    hLabel.config(text="Health: 100")

    # Clear previous widgets in Puzzle and history frames
    for widget in pFrame.winfo_children():
        widget.destroy()
    for widget in gFrame.winfo_children():
        widget.destroy()

    # Setting default screen to open to the rules
    if key == "0: Rules":
        hLabel.config(text="")  # Hide health bar when rules open
        subButton.config(state=tk.DISABLED)  # Turn off submission button

        rulestxt = PUZLVL["0: Rules"][0]
        tk.Label(pFrame, text=rulestxt, font=("Arial", 9), justify="left").pack(pady=10)
        return

    # Re-enable submit button and health bar
    subButton.config(state=tk.NORMAL)
    
    # Select random quote pair based on difficulty
    cQuote, eQuote = random.choice(PUZLVL[key])
    correct = cQuote.upper()
    encrypted = eQuote.upper()

    corAns = [ch for ch in correct if ch.isalpha()]
    entries.clear()

    # Build Puzzle grid dynamically (scaled down)
    for i, ch in enumerate(encrypted):
        label = tk.Label(pFrame, text=ch, font=("Arial", 9, "bold"))
        label.grid(row=0, column=i, padx=1)

        if ch.isalpha():
            entry = tk.Entry(pFrame, width=2, font=("Arial", 8), justify="center", validate="key", validatecommand=vcmd)
            entry.grid(row=1, column=i, padx=1)
            entries.append((i, entry))
        else:
            tk.Label(pFrame, text=ch, font=("Arial", 9)).grid(row=1, column=i)

# User Guesses
def subGuess():
    wrong = False
    corGuesses = {}

    # Check each entry box against its exact string position
    for ogIndx, eWidget in entries:
        guess = eWidget.get().upper()
        reLet = correct[ogIndx]

        if guess == reLet:
            eWidget.config(bg="OliveDrab1")
            if guess:
                corGuesses[reLet] = guess
        else:
            eWidget.config(bg="tomato")
            if guess:
                wrong = True

    # Deduct health if any wrong guess was made
    if wrong:
        hLvl(10)

    # Autofill matching letters in all positions
    for reLet, uGuess in corGuesses.items():
        for ogIndx, eWidget in entries:
            if correct[ogIndx] == reLet:
                eWidget.delete(0, tk.END)
                eWidget.insert(0, uGuess)
                eWidget.config(bg="OliveDrab1")

    # Record history row
    row = tk.Frame(gFrame)
    row.pack(pady=1)
    for colIdx, (ogIndx, eWidget) in enumerate(entries):
        tk.Label(row, text=eWidget.get().upper() or "-", width=2, font=("Arial", 8), justify="center", bg=eWidget.cget("bg")).grid(row=0, column=colIdx, padx=1)

def restartGame():
    loadLevel(lvlselect.get())

# User Select Menu Design
menuFrame = tk.Frame(window)
menuFrame.pack(pady=2)

mLabel = tk.Label(menuFrame, text="Category: ", font=("Arial", 9, "bold"))
mLabel.pack(side=tk.LEFT, padx=3)

lvlDrop = tk.OptionMenu(menuFrame, lvlselect, *PUZLVL.keys(), command=loadLevel)
lvlDrop.config(font=("Arial", 9))
lvlDrop.pack(side=tk.LEFT)

# Health & Headers
hLabel = tk.Label(window, text="Health: 100", font=("Arial", 10, "bold"))
hLabel.pack(pady=2)

eLabel = tk.Label(window, text="Cryptogram Board", font=("Arial", 10, "italic"))
eLabel.pack(pady=2)

pFrame = tk.Frame(window)
pFrame.pack(pady=5)

# Action Buttons
btnFrame = tk.Frame(window)
btnFrame.pack(pady=3)

subButton = tk.Button(btnFrame, text="Submit Entry", font=("Arial", 9), command=subGuess)
subButton.pack(side=tk.LEFT, padx=3)

restartButton = tk.Button(btnFrame, text="Restart Level", font=("Arial", 9), command=restartGame)
restartButton.pack(side=tk.LEFT, padx=3)

gFrame = tk.Frame(window)
gFrame.pack(pady=5)

# Initialize game with default level (0: Rules)
loadLevel(lvlselect.get())

window.mainloop()